In [1]:
import os
from ftplib import FTP
from pathlib import Path
import zipfile
import polars as pl

In [2]:
# Estrutura de diretórios para armazenar os dados brutos e os convertidos
nome_pasta = 'cnes_2024'
base_dir = Path(nome_pasta)
raw_dir = base_dir / "raw"
parquet_dir = base_dir / "parquet"

# Criar os diretórios, se ainda não existirem
raw_dir.mkdir(parents=True, exist_ok=True)
parquet_dir.mkdir(parents=True, exist_ok=True)

base_dir, raw_dir, parquet_dir

(WindowsPath('cnes_2024'),
 WindowsPath('cnes_2024/raw'),
 WindowsPath('cnes_2024/parquet'))

In [ ]:
# 📁 Pastas locais
RAW_DIR = f'{nome_pasta}/raw'
PARQUET_DIR = f'{nome_pasta}/parquet'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)
ano = '2024'

# 📦 Tabelas de interesse (nomes de prefixo nos arquivos)
TABELAS_INTERESSE = [
    'rlEstabComplementar',
    'tbAtributo',
    'tbEstabelecimento',
    'tbMunicipio',
    'tbEstado',
    'tbTipoUnidade',
    'tbTipoEstabelecimento',
    'tbAtividade',
    'tbAtividadeProfissional',
]

# 📅 Meses a baixar
MESES = [f'{mes:02d}' for mes in range(1, 13)]

# 🔽 Baixa e extrai arquivos CSV específicos do ZIP

def baixar_e_extrair(mes, ano=ano):
    competencia = f'{ano}{mes}'
    arquivo_zip = f'BASE_DE_DADOS_CNES_{competencia}.zip'
    print(f'🔽 Baixando {arquivo_zip}...')

    ftp = FTP('ftp.datasus.gov.br')
    ftp.login()
    ftp.cwd('cnes')  # Diretório correto de acordo com sua observação

    caminho_local = os.path.join(RAW_DIR, arquivo_zip)
    with open(caminho_local, 'wb') as f:
        ftp.retrbinary(f'RETR {arquivo_zip}', f.write)

    ftp.quit()

    # Extrai apenas os CSVs de interesse
    print(f'📦 Extraindo arquivos de interesse de {arquivo_zip}...')
    with zipfile.ZipFile(caminho_local, 'r') as zip_ref:
        for nome_arquivo in zip_ref.namelist():
            for base in TABELAS_INTERESSE:
                if nome_arquivo.lower().startswith(base.lower()) and nome_arquivo.lower().endswith('.csv'):
                    zip_ref.extract(nome_arquivo, RAW_DIR)

# 🧪 Converte arquivos CSV extraídos para Parquet usando Polars

def converter_csv_para_parquet():
    for nome_arquivo in os.listdir(RAW_DIR):
        if nome_arquivo.lower().endswith('.csv'):
            caminho_csv = os.path.join(RAW_DIR, nome_arquivo)
            nome_base = nome_arquivo.split('.')[0]
            nome_base_sem_data = ''.join(filter(str.isalpha, nome_base))

            if nome_base_sem_data.lower() not in [t.lower() for t in TABELAS_INTERESSE]:
                continue

            print(f'📄 Convertendo {nome_arquivo} → {nome_base}.parquet')

            df = pl.read_csv(
                caminho_csv, 
                encoding='latin1', 
                separator=';', 
                low_memory=True,
                schema_overrides={
                    'CO_UNIDADE': pl.Utf8
                }
            )
            df.write_parquet(os.path.join(PARQUET_DIR, f'{nome_base}.parquet'))

# 🚀 Execução
for mes in MESES:
    baixar_e_extrair(mes)

converter_csv_para_parquet()
print("✅ Processo finalizado com sucesso!")

🔽 Baixando BASE_DE_DADOS_CNES_202401.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202401.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202402.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202402.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202403.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202403.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202404.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202404.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202405.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202405.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202406.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202406.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202407.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202407.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202408.zip...
📦 Extraindo arquivos de interesse de BASE_DE_DADOS_CNES_202408.zip...
🔽 Baixando BASE_DE_DADOS_CNES_202409.zip...
📦 Extraindo arquivos de interesse de BASE_DE

ComputeError: could not parse `"CE00000000000000006580724000138"` as dtype `i64` at column 'CO_UNIDADE' (column number 1)

The current offset in the file is 17318 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `"CE00000000000000006580724000138"` to the `null_values` list.

Original error: ```remaining bytes non-empty```

In [4]:
def converter_csv_para_parquet():
    for nome_arquivo in os.listdir(RAW_DIR):
        if nome_arquivo.lower().endswith('.csv'):
            caminho_csv = os.path.join(RAW_DIR, nome_arquivo)
            nome_base = nome_arquivo.split('.')[0]
            nome_base_sem_data = ''.join(filter(str.isalpha, nome_base))

            if nome_base_sem_data.lower() not in [t.lower() for t in TABELAS_INTERESSE]:
                continue

            print(f'📄 Convertendo {nome_arquivo} → {nome_base}.parquet')

            df = pl.read_csv(
                caminho_csv, 
                encoding='latin1', 
                separator=';', 
                low_memory=True,
                schema_overrides={
                    'CO_UNIDADE': pl.Utf8
                }
            )
            df.write_parquet(os.path.join(PARQUET_DIR, f'{nome_base}.parquet'))

converter_csv_para_parquet()
print("✅ Processo finalizado com sucesso!")


📄 Convertendo rlEstabComplementar202401.csv → rlEstabComplementar202401.parquet
📄 Convertendo rlEstabComplementar202402.csv → rlEstabComplementar202402.parquet
📄 Convertendo rlEstabComplementar202403.csv → rlEstabComplementar202403.parquet
📄 Convertendo rlEstabComplementar202404.csv → rlEstabComplementar202404.parquet
📄 Convertendo rlEstabComplementar202405.csv → rlEstabComplementar202405.parquet
📄 Convertendo rlEstabComplementar202406.csv → rlEstabComplementar202406.parquet
📄 Convertendo rlEstabComplementar202407.csv → rlEstabComplementar202407.parquet
📄 Convertendo rlEstabComplementar202408.csv → rlEstabComplementar202408.parquet
📄 Convertendo rlEstabComplementar202409.csv → rlEstabComplementar202409.parquet
📄 Convertendo rlEstabComplementar202410.csv → rlEstabComplementar202410.parquet
📄 Convertendo rlEstabComplementar202411.csv → rlEstabComplementar202411.parquet
📄 Convertendo rlEstabComplementar202412.csv → rlEstabComplementar202412.parquet
📄 Convertendo tbAtividade202401.csv → tb

In [5]:
# Análise de registro das colunas de cada tabela

arquivos = [
    'tbEstabelecimento202412',
    'rlEstabComplementar202412',
    'tbAtributo202412',
    'tbMunicipio202412',
    'tbEstado202412',
    'tbTipoUnidade202412',
    'tbTipoEstabelecimento202412',
    'tbAtividade202412',
    'tbAtividadeProfissional202412'
]

for arquivo in arquivos:
    df = pl.read_parquet(os.path.join(PARQUET_DIR, f'{arquivo}.parquet'))
    print(f'🔍 Exibindo a contagem de registros de todas as colunas da tabela {arquivo}.parquet -> Shape: ', df.shape, '\n')
    for col in df.columns:
        print(f'{df[col].value_counts()}')
    print('\n' + '-'*100 + '\n')


🔍 Exibindo a contagem de registros de todas as colunas da tabela tbEstabelecimento202412.parquet -> Shape:  (557329, 56) 

shape: (557_329, 2)
┌───────────────┬───────┐
│ CO_UNIDADE    ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ 2925206609708 ┆ 1     │
│ 4107204272293 ┆ 1     │
│ 3304550739359 ┆ 1     │
│ 3515503735613 ┆ 1     │
│ 4310207115334 ┆ 1     │
│ …             ┆ …     │
│ 3514409772278 ┆ 1     │
│ 3549804293878 ┆ 1     │
│ 5002707608543 ┆ 1     │
│ 2211002360845 ┆ 1     │
│ 2901009055932 ┆ 1     │
└───────────────┴───────┘
shape: (557_329, 2)
┌─────────┬───────┐
│ CO_CNES ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 110116  ┆ 1     │
│ 6493025 ┆ 1     │
│ 4073371 ┆ 1     │
│ 510505  ┆ 1     │
│ 2974118 ┆ 1     │
│ …       ┆ …     │
│ 4609522 ┆ 1     │
│ 7019807 ┆ 1     │
│ 7620438 ┆ 1     │
│ 9971041 ┆ 1     │
│ 656666  ┆ 1     │
└─────────┴───────┘
shape: (7_270, 2)
┌─────────────────────┬───────┐
│ NU_

In [6]:
# Diretório dos arquivos Parquet
PARQUET_DIR = f'{nome_pasta}/parquet'
arquivos = os.listdir(PARQUET_DIR)

# Criação de uma coluna de data para registrar a primeira data de cada registro nas cada tabela
def adicionar_coluna_data(df, nome_arquivo):
    # Adiciona o ano_mes como uma nova coluna
    ano_mes = nome_arquivo.split('.')[0][-6:]
    ano = ano_mes[:4]
    mes = ano_mes[-2:]
    df = df.with_columns(
        pl.concat_str(
            pl.lit(ano),
            pl.lit('-'),
            pl.lit(mes),
            pl.lit('-01')
        ).alias('data_competencia')
    )
    return df

# Aplicando a função para adicionar a coluna de data
for arquivo in arquivos:
    df = pl.read_parquet(os.path.join(PARQUET_DIR, f'{arquivo}'))
    df = adicionar_coluna_data(df, arquivo)
    df.write_parquet(os.path.join(PARQUET_DIR, f'{arquivo}'))

In [15]:
TABELAS_INTERESSE = [
    'rlEstabComplementar',
    'tbAtributo',
    'tbEstabelecimento',
    'tbMunicipio',
    'tbEstado',
    'tbTipoUnidade',
    'tbTipoEstabelecimento',
    'tbAtividade',
    'tbAtividadeProfissional',
]

for file in os.listdir('cnes_2024/parquet'):
    for tabela in TABELAS_INTERESSE:
        df = pl.read_parquet(os.path.join('cnes_2024/parquet', file))
        print(f'🔍 Exibindo a contagem de registros de todas as colunas da tabela {file} -> Shape: ', df.shape, '\n')

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibindo a contagem de registros de todas as colunas da tabela rlEstabComplementar202401.parquet -> Shape:  (59133, 11) 

🔍 Exibin

In [16]:
df = pl.read_parquet(os.path.join('cnes_2024/parquet', 'tbEstabelecimento202412.parquet'))
df_2 = pl.read_parquet(os.path.join('cnes_2024/parquet', 'tbEstabelecimento202405.parquet'))

In [18]:
set(df.columns) - set(df_2.columns)

{'ST_COWORKING'}

In [24]:
# Concatena arquivos parquet por tabela base (sem data)
def concatenar_parquets_por_tabela():
    print("🔗 Concatenando arquivos por tabela base...")
    arquivos = os.listdir(PARQUET_DIR)
    tabelas_encontradas = {}

    # Agrupa arquivos por base
    for arquivo in arquivos:
        if not arquivo.endswith('.parquet'):
            continue
        nome_base = ''.join(filter(str.isalpha, arquivo.replace('.parquet', '')))
        tabelas_encontradas.setdefault(nome_base.lower(), []).append(arquivo)

    for base, arquivos_base in tabelas_encontradas.items():
        caminhos = [os.path.join(PARQUET_DIR, f) for f in arquivos_base]
        print(f'Concatenando {len(caminhos)} arquivos da tabela {base}...')

        if 'tbestabelecimento' in base.lower():
            dfs = []
            for c in caminhos:
                df = pl.read_parquet(c)
                if 'ST_COWORKING' in df.columns:
                    df = df.drop('ST_COWORKING')
                dfs.append(df)
            df_final = pl.concat(dfs, how='vertical_relaxed')

        else:
            dfs = [pl.read_parquet(c) for c in caminhos]
            df_final = pl.concat(dfs, how='vertical_relaxed')

        df_final.write_parquet(os.path.join('cnes_final', f'{base}_2024.parquet'))
        print(f'Tabela {base}_2024.parquet salva com sucesso!')

concatenar_parquets_por_tabela()

🔗 Concatenando arquivos por tabela base...
Concatenando 12 arquivos da tabela rlestabcomplementar...
Tabela rlestabcomplementar_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbatividade...
Tabela tbatividade_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbatividadeprofissional...
Tabela tbatividadeprofissional_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbatributo...
Tabela tbatributo_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbestabelecimento...
Tabela tbestabelecimento_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbestado...
Tabela tbestado_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbmunicipio...
Tabela tbmunicipio_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbtipoestabelecimento...
Tabela tbtipoestabelecimento_2024.parquet salva com sucesso!
Concatenando 12 arquivos da tabela tbtipounidade...
Tabela tbtipounidade_2024.parquet

In [27]:
# for file in os.listdir('cnes_final'):
#     print(file)
#     df = pl.read_parquet(os.path.join('cnes_final', file))
#     print(df.columns)

#     if 'CO_UNIDADE' in df.columns:
    
#         df_ativo = df['CO_UNIDADE', 'data_competencia'].group_by('CO_UNIDADE').agg(
#             pl.col('data_competencia').first().alias('data_primeiro_registro'),
#             pl.col('data_competencia').last().alias('data_ultimo_registro')
#         )

#         df = df.join(
#             df_ativo,
#             left_on='CO_UNIDADE',
#             right_on='CO_UNIDADE',
#             how='inner'
#         )

#         df = df.sort('data_ultimo_registro')

#         df_lazy = df.lazy()
#         df_deduplicado = df_lazy.unique(subset=["CO_UNIDADE"], keep='last').collect()

#         df = pl.read_parquet(f'cnes_2022/parquet/{file.replace('_2022', '202212')}')

#         print(f'🔍 Exibindo a contagem de registros da tabela {file} deduplicada -> Shape: ', df_deduplicado.shape, '\n')
#         print(f'🔍 Exibindo a contagem de registros da tabela {file} de dezembro -> Shape: ', df.shape, '\n')            

In [ ]:
estab_complementar = pl.read_parquet('cnes_final/rlestabcomplementar_2022.parquet')
atividade = pl.read_parquet('cnes_final/tbatividade_2022.parquet') #ok
atividade_profissional = pl.read_parquet('cnes_final/tbatividadeprofissional_2022.parquet') 
municipio = pl.read_parquet('cnes_final/tbmunicipio_2022.parquet') #ok
estado = pl.read_parquet('cnes_final/tbestado_2022.parquet') #ok
tipo_unidade = pl.read_parquet('cnes_final/tbtipounidade_2022.parquet') #ok
tipo_estabelecimento = pl.read_parquet('cnes_final/tbtipoestabelecimento_2022.parquet') # ok
atributo = pl.read_parquet('cnes_final/tbatributo_2022.parquet') #ok

In [28]:
estabelecimento = pl.read_parquet('cnes_final/tbEstabelecimento_2024.parquet')

In [29]:
estabelecimento = estabelecimento.select(
    [
        'CO_UNIDADE', 'CO_CNES', 'NU_CNPJ_MANTENEDORA', 'TP_PFPJ',
        'NIVEL_DEP', 'NO_RAZAO_SOCIAL', 'NO_FANTASIA', 'NO_LOGRADOURO',
        'NU_ENDERECO', 'NO_COMPLEMENTO', 'NO_BAIRRO', 'CO_CEP',
        'NU_CNPJ', 'CO_ATIVIDADE', 'TP_UNIDADE', 'CO_TURNO_ATENDIMENTO',
        'CO_ESTADO_GESTOR', 'CO_MUNICIPIO_GESTOR', 'CO_MOTIVO_DESAB',
        'TP_ESTAB_SEMPRE_ABERTO', 'CO_TIPO_UNIDADE', 'CO_TIPO_ESTABELECIMENTO',
        'CO_ATIVIDADE_PRINCIPAL', 'data_competencia'
    ]
)

estabelecimento_ativo = estabelecimento['CO_CNES', 'data_competencia'].group_by('CO_CNES').agg(
    pl.col('data_competencia').first().alias('data_primeiro_registro'),
    pl.col('data_competencia').last().alias('data_ultimo_registro')
)

estabelecimento = estabelecimento.join(
    estabelecimento_ativo,
    left_on='CO_CNES',
    right_on='CO_CNES',
    how='inner'
)

estabelecimento = estabelecimento.sort('data_ultimo_registro')

In [30]:
estabelecimento_ativo.shape

(557342, 3)

In [31]:
estabelecimento_lazy = estabelecimento.lazy()
estabelecimento_deduplicado = estabelecimento_lazy.unique(subset=["CO_CNES"], keep='last').collect()

In [37]:
estabelecimento_deduplicado.filter(
    (pl.col('CO_MOTIVO_DESAB') == '')
)['CO_CNES'].n_unique()

445308

In [34]:
df = pl.read_parquet('cnes_2024/parquet/tbEstabelecimento202412.parquet')

In [38]:
df.shape

(557329, 57)

In [40]:
estabelecimento_deduplicado.filter(
    (pl.col('CO_MOTIVO_DESAB') == '')
)['CO_CNES'].n_unique()

445308

In [41]:
estabelecimento_deduplicado

CO_UNIDADE,CO_CNES,NU_CNPJ_MANTENEDORA,TP_PFPJ,NIVEL_DEP,NO_RAZAO_SOCIAL,NO_FANTASIA,NO_LOGRADOURO,NU_ENDERECO,NO_COMPLEMENTO,NO_BAIRRO,CO_CEP,NU_CNPJ,CO_ATIVIDADE,TP_UNIDADE,CO_TURNO_ATENDIMENTO,CO_ESTADO_GESTOR,CO_MUNICIPIO_GESTOR,CO_MOTIVO_DESAB,TP_ESTAB_SEMPRE_ABERTO,CO_TIPO_UNIDADE,CO_TIPO_ESTABELECIMENTO,CO_ATIVIDADE_PRINCIPAL,data_competencia,data_primeiro_registro,data_ultimo_registro
str,i64,str,i64,i64,str,str,str,str,str,str,i64,str,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str
"""4120709487085""",9487085,"""""",3,1,"""MARIA VALDINEIA PIRATELLO ZANI…","""NOVA FACE BELEZA E ESTETICA""","""RUA JOAO AMBROSIO""","""20""","""""","""CENTRO""",86450000,"""11358290000166""",4,22,3,41,412070,"""""","""N""","""""","""016""","""001""","""2024-12-01""","""2024-01-01""","""2024-12-01"""
"""5220454569873""",4569873,"""""",3,1,"""DROGARIA SAO PAULO S A""","""DROGARIAS PACHECO""","""DOM EMANUEL""","""S/N""","""SETOR 105 QUADRA4 I""","""JARDIM TODOS OS SANT""",75261432,"""61412110129000""",4,43,4,52,522045,"""""","""N""","""""","""009""","""008""","""2024-12-01""","""2024-04-01""","""2024-12-01"""
"""3540004927990""",4927990,"""""",3,1,"""UNIDADE RADIOLOGICA DE TUPA LT…","""CLINICA IMAGEM DE POMPEIA""","""SHINJI HAMAZAKI""","""S/N""","""""","""FLANDRIA""",17582130,"""51517738000323""",4,4,6,35,354000,"""""","""S""","""""","""018""","""002""","""2024-12-01""","""2024-11-01""","""2024-12-01"""
"""1200702001225""",2001225,"""04018560000124""",3,3,"""PREFEITURA MUNICIPAL DE XAPURI""","""UNIDADE DE SAUDE DA FAMILIA JO…","""RUA DA CERAMICA""","""S/N""","""""","""SIBERIA""",69930000,"""""",4,2,4,12,120070,"""""","""N""","""""","""001""","""012""","""2024-12-01""","""2024-01-01""","""2024-12-01"""
"""3550304657411""",4657411,"""""",3,1,"""FISIOKIDS FISIOTERAPIA LTDA""","""FISIOKIDS FISIOTERAPIA""","""RUA PARA""","""76""","""SALA 24""","""CONSOLACAO""",1243020,"""10552926000143""",4,22,3,35,355030,"""""","""N""","""""","""016""","""001""","""2024-12-01""","""2024-05-01""","""2024-12-01"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""5205106957722""",6957722,"""""",3,1,"""OPIMED DO BRASIL LTDA""","""WIDEX APARELHOS AUDITIVOS""","""FARID MIGUEL SAFATLE""","""59""","""SALA 01""","""SETOR CENTRAL""",75701040,"""01191654000617""",4,22,3,52,520510,"""""","""N""","""""","""016""","""001""","""2024-12-01""","""2024-01-01""","""2024-12-01"""
"""3304556898831""",6898831,"""""",3,1,"""RRM REDE RIO DE MEDICINA LTDA""","""RRM CENTRO MEDICO COPACABANA""","""RUA SIQUEIRA CAMPOS""","""00121""","""GRP 402 GRP 901 GRP""","""COPACABANA""",22031071,"""33710096000483""",4,36,4,33,330455,"""""","""N""","""""","""016""","""001""","""2024-12-01""","""2024-01-01""","""2024-12-01"""
"""3167209953116""",9953116,"""""",3,1,"""MARCIO LANZA JUNIOR CLINICA DE…","""MARCIO LANZA JUNIOR CLINICA DE…","""RUA TEOFILO OTONI""","""315""","""SALA 07""","""CENTRO""",35700007,"""34791271000124""",4,22,3,31,316720,"""""","""N""","""""","""016""","""001""","""2024-12-01""","""2024-01-01""","""2024-12-01"""


In [42]:
estab_complementar = pl.read_parquet('cnes_final/rlestabcomplementar_2022.parquet')
atividade = pl.read_parquet('cnes_final/tbatividade_2022.parquet') #ok
atividade_profissional = pl.read_parquet('cnes_final/tbatividadeprofissional_2022.parquet') 
municipio = pl.read_parquet('cnes_final/tbmunicipio_2022.parquet') #ok
estado = pl.read_parquet('cnes_final/tbestado_2022.parquet') #ok
tipo_unidade = pl.read_parquet('cnes_final/tbtipounidade_2022.parquet') #ok
tipo_estabelecimento = pl.read_parquet('cnes_final/tbtipoestabelecimento_2022.parquet') # ok
atributo = pl.read_parquet('cnes_final/tbatributo_2022.parquet') #ok

In [44]:
estab_complementar

CO_UNIDADE,CO_LEITO,CO_TIPO_LEITO,TP_ALTACOMP,QT_EXIST,QT_CONTR,QT_SUS,"TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')",CO_USUARIO,"TO_CHAR(DT_ATUALIZACAO_ORIGEM,'DD/MM/YYYY')",data_competencia
str,i64,str,str,i64,str,i64,str,str,str,str
"""3107405661943""",11,"""1 ""","""""",1,"""""",0,"""15/04/2019""","""SAOZINHA""","""22/07/2012""","""2022-01-01"""
"""3107405661943""",15,"""1 ""","""""",1,"""""",0,"""15/04/2019""","""SAOZINHA""","""22/07/2012""","""2022-01-01"""
"""3136703126277""",7,"""7 ""","""""",2,"""""",0,"""16/08/2019""","""CNESSCC""","""22/07/2012""","""2022-01-01"""
"""2918409192484""",13,"""1 ""","""""",6,"""""",6,"""31/03/2017""","""ROSANGELA""","""03/04/2017""","""2022-01-01"""
"""2403807206933""",33,"""2 ""","""""",5,"""""",5,"""18/04/2016""","""FLORA""","""05/04/2013""","""2022-01-01"""
…,…,…,…,…,…,…,…,…,…,…
"""4209106048692""",5,"""1 ""","""""",1,"""""",1,"""23/09/2022""","""420910""","""27/06/2018""","""2022-12-01"""
"""2907802532522""",45,"""5 ""","""""",6,"""""",6,"""01/12/2022""","""REGINA""","""11/04/2005""","""2022-12-01"""
"""2408002503654""",33,"""2 ""","""""",18,"""""",18,"""05/01/2023""","""FABRICIA""","""11/04/2005""","""2022-12-01"""


In [ ]:
estab_complementar.filter(
    pl.col('CO_UNIDADE').is_in(estabelecimento_deduplicado['CO_UNIDADE'])
).sort('data_competencia').unique(subset=['CO_UNIDADE','CO_LEITO','CO_TIPO_LEITO','TP_ALTACOMP','QT_EXIST','QT_CONTR','QT_SUS'], keep='last').shape

(70990, 11)

In [150]:
pl.read_parquet('cnes_2022/parquet/tbAtributo202212.parquet').shape

(24, 4)

In [151]:
atributo.sort('data_competencia').unique(subset=atributo.columns[:-2], keep='last').shape

(24, 4)